# Train a tiny character-level language model

Train a small causal language model on characters, from scratch, fast enough for a CPU.

```python
def train_char_lm(text, block_size=32):
    # Your code
    return model
```

- `text` is a Python string of about 20,000 characters made by the generator below (with a seed you do not know).
- Return a trained `nn.Module` with an attribute `model.stoi`, a dict mapping every character of `text` to a distinct id `0, ..., V-1`, and a `forward(idx)` that takes an `int64` tensor of shape `(B, T)` with `T <= block_size` and returns next-token **logits** of shape `(B, T, V)`: position `t` predicts character `t + 1` from characters `0..t` only.

The grader calls `model.eval()`, encodes a held-out text from the same generator with your `stoi`, cuts it into rows of `block_size + 1` characters, and computes the mean cross-entropy of your logits for characters `1..block_size` of each row. It must be **at most 0.85 nats per character** (0.90 for a second hidden text with `block_size = 16`). For scale: a bigram model gets about 1.12, and the best possible model, which knows that each word is drawn uniformly from the 15 below, gets about 0.73 ($\ln 15$ nats per word of 3.8 characters, space included, is 0.713, plus a little at the start of each row). It also checks that changing a later character never changes the logits at earlier positions.

```python
import random
WORDS = ["the", "cat", "sat", "on", "mat", "dog", "ran", "to", "a", "red", "hat", "and", "big", "sun", "set"]

def make_text(n_words, seed):
    r = random.Random(seed)
    return " ".join(r.choice(WORDS) for _ in range(n_words)) + " "

train_text = make_text(5000, seed=1)      # about 18,500 characters
```

**Suggested model.** A pre-norm transformer with token and learned position embeddings, $d = 64$, 2 blocks, 4 heads, causal self-attention you write yourself (u16-p01), a final layer norm and a linear output layer: about $10^5$ parameters. Train on random windows of `block_size + 1` characters, batch 64, AdamW with learning rate $3 \times 10^{-3}$, for a few hundred steps. Set `torch.manual_seed` inside your function so the result is reproducible.

**Restrictions.** Only `import torch` (anything inside it), `import math` and `import random`. You may not use `nn.MultiheadAttention`, any `nn.Transformer...` module or `F.scaled_dot_product_attention`.

**Constraints.** Time limit 40 seconds per call to `train_char_lm` on Colab's CPU. The grader runs on a faster CPU and allows 20 seconds there. The suggested model needs about 5 seconds.

**The full-size version (not graded).** The same code trains a real character model on a GPU. In Colab choose Runtime, Change runtime type, L4 GPU; set `device = "cuda"` and move the model (`model.to(device)`) and every batch (`x.to(device)`) to it. Use about 1 MB of plain text (for example a public-domain book), $d = 384$, 6 blocks, 6 heads, `block_size = 256`, dropout 0.2, batch 64, AdamW with learning rate $10^{-3}$ and about 5000 steps. The nanoGPT project reports a validation loss near 1.47 nats per character with these settings on its Shakespeare dataset. Generate with the sampling loop of u16-l04.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem5_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem5_submission.py
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F


def train_char_lm(text, block_size=32):
    torch.manual_seed(0)
    stoi = {c: i for i, c in enumerate(sorted(set(text)))}
    # Your code: build the model, train it, set model.stoi = stoi
    return model


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem5_submission as submission
importlib.reload(submission)
# e.g. submission.train_char_lm(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwlz2ziW/itYde2YzMiM7BzbpXR61+k4Pdlyjko83bNlu9iQCMkYU6CGoGJ7PP7v+70HgAQlOYnTvZtUWSIIvAvvBnQz0MVgLAarvae7y9GTwVAMFqqRGLrZfNPoplQ0dlxLbYQUjTbXYnouazltVL1bqk+qFKU085WcK7GoClXSulJZW5kArBw9ZjTarBplMbr/BI+Fns30dFU21xh5jAFlmpq+DxpClhOWvFzwyqpYlSo3csHELOtqUqrFk9yuJgttrQYqzJJlWV2qIteLZVU3hOdk0FT19JxByIY/a2mKajE4w9eJNEYVPO0NyNDnShYHTQMyPEAwbeysqheqPjRT8FYfyWtVb3+1NvpS3bnAv1obpUc7lSUYKKomB4/FatrksiXIk5yXVbUk5maytIr2SNkmb/RCVasmpxd7+yMalnMngnKRsUC1mfPWLDKjrrCiulBObAFDNpUrK8t8Ie0Fbz7Jrl3LeJkIW63qKW9EVeu5NpK3fCnrZh+DTb1St3ie6ZL3+maAfTTQh2zJm3vK//+iyqWqrWjOZSP8fop5LQsadBsokt9mdbUQfnkYffBbmp2aU3PgpwttxbQyxMPDldGNffgDffz40EPF81IXPz50s0FEJl43olAzDaiChCdmKzMlAVhBGlbwYP7A0dbIC4Vv6tR0yiacPorfMPRbJg7/dvDm/dHhR1Fq2xBLyoElYFbY8+rSCCz6sDIi+aStBlXpM/GRwDWiXgEtrKi+5kVg7ZXU5arGSgjeKkGP4lI357A+wLNkZ5e1pk0T0BzxQpmxsPJaXBK5l9LibWXmQ2EIqihWi6U410WB6dosV40l6bldMF6k02p53T6QJpF8/aPB+msBqGZJo9+Jn6pSTsRP7/9qhayVsGRzNcnKECk7VrzZF+/raihsJcoKCk0QoT9WLFYQykSRUCGiiqxf43WpSQ4T8H+OrZzBsVR1Rpj2sxFtrhTzFe1/4jBbwCrVbnNew1yFXSpsGGbV1Wp+Xl5DW8qZAC2gYlrVJOhakdAboWdEIB7ZO3g+vN2QUD6+Pzx8mb86+On43QfxnLATw6dmWkpgp21IDq+makmako5PjcC/JV65WVAoSFwVCXSxGIqFnYc5wGsqErIp/Aj96zY3obkdkNwsk6uwFg4xWhN2iOwyGp2Bf21sI81UJVdD9z47VsZWdRotZ7SqWdVGXGUFfP70PEmz6XKFv7zNSeomK2ZTvGZ0h3Vd1REUx3IEyywzaWVdy2vQ3fFhz+VSJfOqGUIpDf2Ffj4/HUCz4WxPB4EyC1k3q2WpEuIc89OMl3paWKaYEyYRrHQoZqeDGwJ4iw23Dpe4sfA76mqpEJoKcRPNvwW6jrJpWdnPUTYUdVOVz/fU7tOhkP7r94FeWNYlSG6Jlba5BqOzspJEGI0zzv6LiJm5449YunRf72DHMeCnp9t4C6+YvRYBNkRbuDfdYDXoKMskFfAUpIXxu0v/LkZPnhT+njyhoTXazFrgXpFpv8vSCxHC8OKiP15c9CdWPFXXkBgLgohbyCv6kBObzMWuuEzTlGCDUf1PJRSimhiR8W0xlb6c2NGJT7KEfxBJKes5uV1FCitu8DHOHs1v097eq3+sZPkVWtk3KjeViDYFa/owZYmCCETP2r8/HfB2dEC2qEukIN2c/3O1iFWD6c+9HEDchjHFUm0h0K6M1ymGMhC1zHwHA874Bq/+re6RRrMw1NsMbXM2Ed4O+mbv3A9GGO1It2SDfhqFKDzgNMs5b8zzHjlzuD/aOY91J0zaCVD7lObWyCUiefP/65rLyqh7OmWgDFp5RSrJsX2QIiQWbL19epJCTyFISlzSmJ4+cz3qCGBSKLV8foxML1IwT+AxxLdO3iaAtBdCaCgjmPzuqid4bEsih2Ly7XKXXyV3N8XZBeP7VrFLEjvDsX1X4BFJh8Qmk3Sbr8Hyzs9sXd76iYn/ynvbt2ziYOi8HTTbxHsV4DCEKHmp8sUKRoHsJpmZoXgAdwrLePDg4pK+BUpc6vgTIoCYGcY8owSVeGioMDRzmJdEkYhFqwWScoQQsSzlVGUh6+SoT/YEv3jSWZZMOZ+VtIAwnrmZyM4oeJhknSD3mpZoKDIYtiktVUhlVC0R3/6pl4lbwzh6Cs7+pFUu70SuUdgIqs0owdczTZklpcwdL9/daPFnsXf7TFxW9QVl9ZKVV9D2IR9t/aUXMojvJExJvDaJVZQKgqqvlfKzWMTwrnHKvSWT5jIY8dAVkywgznV32pVI3l0NRajeAURNJRgKpIUlljgPfyOnYgoEtARMuqybqhVO7Ck5QHWxO1kV8KQ0zDk40nJNBdRqCXMSzaWeKgIICWlE6GrmLRg0B4laViEamSFfovBNELBihXeZOKYXusYwmCl3CAXFBYeMJKyKnlo1I8omASFDdTnLGYyqg7f5gioVzfbFyFEan454fp8Lv4nioYjLhk4lc1LFmqwh2Y/1DpsIND8895DWXNEENc1FNyT3h+JiH9j6/pFJHq4N9jj5siiYThLEPothPxpmMaBiSwoKsXeII843Y4bYjpqqYr1B8V8042x/dossH1qmnvmqq1ViViaYHQMSK0P1/I0XjV/HNSQKu7DFW/4xSlSWKINnUK05JbCuyLzx+8SAyDo+a6C1mcM6VRE2rCtwXNcI0p5JqF/eTjzlDsrKSqmrnDsMvS7HzzQCCwBlXfOA+2jTc10WZHVT6oygYNZkCzTzvz++ewtaDVuObQpQyLa6vG7OMbL7WgQ84gff4sgLXf8ofuhw4CWeYZX/4rHmx42SH8YWlf/Y2Kum1JN2yH1gJFs1uuyGq/br323VNQmWsjmPl1s9N7JbZq/tWnshPNSICxM5vWhHoMXUbLKu12CnGmwa1bo7DFzoZrdUmCb2sqfZns/p7Jj6RAvqOw7ZpaC68MEnOL+l1JDy/NQEHNlMl9Do8JicDkB2VSuqAH2nBZlo9uCv5sJQD8dWJXVUKu4AAKFe0q5lD3ym+JfDD4ewHC+L7D0+kzynNliepxkcFy1HVreUNSg7NZBKRpMzaKuqm2QEY0D6QGBSANzW++JekPgOsfofciwOH4/2uRN29OvB/3zMXxy8fXv4EhTcnA5y3wTNc2IGjIX9dI/qSk39N+T57tu0WixB6+ng9tQcvXv3Pn/77uXhR4BLoCvZqwq2Sl9+heIq9/XAXptp++IIOeRPgOGePqro4SWSzO7pZ2UoPlf14dUyjVssx64X86UuSy5R6S0S0rHVAv4GYUu1FsulYgAUdyMos5nm03M1vbCJrae0xY3sx9sPzt4pG0awIjlWF044CzuHYJ7BLVIG6Hp8bDa22Wl3h5Mh30qkQNXU2rUVeyHKBaGTs23ZbFMrJHYsJWgJ6msQ2s9DP16jPr/iPJSacmozPzzxZI+hLdwg9tSPyU1aXu5L5Mo4P3OjMvo01e2YvmM2ip9bT6BvqlNIgLwyOGbYyVqnnWR0cuYJnch2nONkE0IOt0PzlWVg0fgE9pC7XqkfJxVHygjb1QW1ppGJ1S5RcIABwc1PqKKhpFtP6GQB7Fz4Q4iMFNsj4IZ5JPOFbBarMh6ZcXcErpp8M4R/KcuLhDZjLXRHaTpNd/rsSoP1qqJNZWlixuSONwOYb6kgX04kT6KkfEGtBfeYARsESZljskDmCXdExRwBXzCxbiPS8fbYGO1FJovC40g3J3d7E83L7LLUtN2E82R372zLwm7zstWyoKx7y9q4SCw/L8ZX8Hrr3GBDsVcsR994B/udQW2RJa/wgsT33yHFdQkC2poUItH1JLBNA9aW3iW8b1rKfN5X6AfBdtZ5X1MIpoMM7augvo088ucA6uJurrZP2oaxC1brONnwM7lcKlMkoePjFp0OnMtz7v0/+/LahoWj4Att3vkodrCaH1iKQanrrawtyCo/741s6HBxQ6edD7oncSAsUsgIJCWJHtSd7j9kAc5KnMrTKBsEzcv+XlF5yg42ifCk6Xbv8meKKBBMAHWzMxQ7DkiwKDLAHeBD3TvfoVbs7fZ+4jb6qWWxhXwbaO+AfdeeIE6ksSTb6bmPEJwKdgGC289Kooj9G3+mz1w1Wl7Ka7vrDoRRySAv1C0gUkoPrTLl9TDgROXShZrEaYFrIvlUipKOa1GqOUrzS0qbBM/MsiyNghmSMlFUtCkO8Lnm+tLtQhIFzT9xcIyisCPXB1+o4b9EEsXSP4leXhipEDDcW3VAgRVePgQ/sekW3QHk9PfssKkCjvZItttlUN5nPndGlFdLl0/2u2003RnZmul9DcNUUUIv/ksE2FSyRnYjEpc5eQwno7NYubewfx8RdFg/xzw7tw2WefTbGHZpEit7j1EMkxmTUtfqXBkuYtvckVf9sfx7zm4/U6nPag0w5XXOmWwCECh0c6qm+hn9O6N2mc5C2WmtuaoQS6gq7Hsu/Ek0T9Dh+JoKNpironmWD+v7HaYJtVVC3ZqhbqarMHkzSVSW5+0LlHw+2STgyDRnHNRnhAcgsFMzKj9VyPmo8mtZ8FnpJbVMqGFFXpZ7k17p8JfSsShr7w3RVrDC0Gw+TuvI98KkExM+IVHR+Qhl/7c3jPWW5gdhL5CJJ20tBk+7XuVSJSvr+aeTNkUMrNw5c/8sKom9pCgBfy7aKY9Cug69xzg1HLKykoVNEibiIakN2QS3IgYEDzGLOhlJcHYonwiiJyaeEKTB3cTnYtMuXMFIQydn9Eylnu0eXWPGqayvpEkTaeBthS2I4Z900M6InDtK0dbQKYMlE7Qnjqozjhb0xFZyFqezoaQL+azlvuMG2thLcNciYXHS5RCbuOlxlHdaElqeMxCtq+xjQ+2T1++C8FyTJ3MfiX/6+Prng6MPb4a+SvdT4c5ybu6GU+DIl/WvL5Es90eBln51zJdgumYVtrPQtYLxuc1IQOl6mhVoVI0mFHUg8/Xx6zeHH/IPhwdHw5a6tTTHLhVpT78VltFoTr0Z19mh7iQfmUQcRbfVnGp0jTlsU+Sq1vCtJpvoPCxGSKgT+rO+EBbj5tkTes0Fw5lT/E2W2IZUnVEPKHfLyPOk3yC4Ub834dsuvZaEV0RvHGfOmzkeyQOzV2OP21TVBZ8piJuwH+NsRB3g4Lo8lhfSqrZD5HogXUvsxdHhaLTXEXAvNu6i2B8MMZl0EkM3xCq/UT4J+kw8as16A/Q2QQW/Qnihz6RRfBKfoAB/PBqNxmffbMh8vbLfx+Gh+H6HmxPOisOJJngZulfplze3k1VRKZc0uct2iC4E4jau2+/Jwvx+Nnk6cJf+yArbkNHeA2z59vcIv2x788j45nfZkoPmZ/nrfr6xFao8N4U8Q7g6GJJ47/Yo1lDSkKBQmTnHTt/It3+StQ04Mt2ohU349ozpRwX2quHcnw5QJMhIZmnc8IKXoENITkkI41ryzOGYtL9ewW85paBshW8RmMDamsMl96zNSsX7N3UB1vnDMQM5eTI+g//mc2FQ67vUgt3j6cCDDpMjbPSWusFU9mwrU7y/dYc+NDTKRrf3ORPbvPjwDVHn90Yefzi36ZW/yaX5TTjpJEeWSon35t2NDQ/eLXa5h7NxdtZlZebhXGbdZ4tEG3fLjLP5IRUQM1mLcDKYxk7AI+fDjbWe9u9kuKWZ0ustd1XuFUv+KFriewZ8XlF8ZQzpYLUKTvBqaHGR3HVQOxSPtoU3n86ehYKsbh3bvUNRu8Dlqs+pdEy63PVzWSk7lUlVlckGYe4dgWp6mkvgmh64dkUoUO4MKXyteSZCweO8W55TfUMHZeNwPEDljjvZbYMFnen69pk7C+7OMN3Vo/gpM6xMxmwMZ6GvIVnVX919zDfkI/GhvzRClLfXzBEYvINXV3KxJEeJrfj13YeX4d25TzwRB71HDF8oBWWPGQaKau6+gC33pfF9R+nHXYvpdHAeloB/92Wi5yHBNcHzNkzM8cGHnw+PQc0o+/7JqcmnEuUOhYHbuKy8UK4aM/llVdONmN7hO6m2O3b/wB/hxD0qYzlkuNZTnU3PK41ownJI1y9heBRpyh0rEZe3/AMHumRMWY5lobcETZBLXOR0LzWQdaEoh0runtfmUjTTx0rHf3zQB7YpFbtDBNFEVhyINTerRe4uvdskvq7BPUjA8reLHiPisefIer+iGTLKLYRuu1zJIMGdyd5wTuNuR/Xgucv84SqZ6eYOBV0Ldc0FhhM1GHr3UW1TaZAdciKPk7xDpUmZqIz+DI00bSjo+qLzFJRfEYsp3UWhB5rA2YRr0jI6/wsEIXmh+4HHlIqO8BOM9pdNdP7rfshx1Wxeo/XdWYKZuYuyCTQL7oSOjhOncaUyjoZ0KxULueywWb4uBXQF1mt4CKELK0Yiy8Qvu3s9/HHLuRt1CnYClSMHzHN6hhK97xQfMJArWxtk3+lG3xI3lHXvyWjU09RCcovG/2CIb1kmJx3HJ1Pnt6cu1bxqgq82ZEokKAJA8WovFQ8fiqSjhG7bBZunW23PGdcJEkPxYGPeWfZJq0tKmrcC4EzOe2J3WyeJM7eymmtOu5nwhNCdjIdi3B18/oKXRG3HWbr9drIDtXbt1B0Q8A1t995f0ya96dM8FL+kwyiHgPFBfJeyLnpm5wmu/G8tRPJiKI5pLTnaTYi3wTTd/XFvd4HUHYaxMxTQ5OjmuEf1KpvW0JScKji6beYZQGjlX3ns7jHJIshsj7P78C51hUp8HYObX7n7vdk29UO5otqfEMxb1WqvjiRptpCGLreSFiZPgtctrtq5FDsoE4Cmru0ZeHzU84QQTAD8fN6Bopt3+Ohfw6ZxYhFaQcobP5KmiX/friFhXqSYUPT9HoyNd58D+AVlpvu/rSoDAXhsv+/HWutAtD/okKzyG6QA2l0vwg9knqyr7A1v4u1YuG3WzfXYXRCmxk8pyclGDjBcHeaGvFNsOGcl61LT9dTKavcLvUQWBd2fY5iCfqrY/4FHnBdxBTXuusula9130R5qC1dGOXKQyRGmrPvFR/tD0ZPZEcUYl+JwZPToxrRSF9x7EASAfwpxNM4eoSQyEhwtY55dhkdcLioY9Y0DeLvJTSu/mJ+eBd3JlCO+pdBHhg0M7veB7Nxzt8N7T78ovP8YiseMZ+9puoUmL7y9p5w7uh8grucgeNt6mm2Sp8Xf3yF5WAdSnSe8AXeD/0P2xKNyWzO4vf1fm/rPvw==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
